# 19-Site Fast Heisenberg-HVA Optimization

This notebook plots the cached-kernel continuous optimization results for the edge-colored Heisenberg HVA. The optimizer applies the same two-qubit Heisenberg gates, but evaluates them through a faster NumPy statevector backend so p-depth searches are practical.

Useful commands from the project root:

```powershell
python scripts/run_19site_heisenberg_hva_fast.py --init weighted --max-layers 2 --method Nelder-Mead --maxiter 35 --restarts 0
python scripts/run_19site_heisenberg_hva_fast.py --init weighted --max-layers 4 --method Nelder-Mead --maxiter 200 --restarts 3 --tag p4_serious
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

EXACT_ENERGY = float(np.load(PROJECT / "results" / "19site_fixed_sz_exact_n9.npz")["energy"])
plt.rcParams["figure.dpi"] = 130

## Load Fast HVA Results

In [ ]:
paths = sorted((PROJECT / "results").glob("19site_heisenberg_hva_fast_*.csv"))
paths = [
    path for path in paths
    if not path.name.endswith("_history.csv") and "smoke" not in path.name
]
if not paths:
    raise FileNotFoundError("Run scripts/run_19site_heisenberg_hva_fast.py first.")

frames = []
for path in paths:
    frame = pd.read_csv(path)
    frame["source_file"] = path.name
    frames.append(frame)
df = pd.concat(frames, ignore_index=True)
df[[
    "source_file",
    "initialization",
    "depth",
    "energy",
    "error_vs_reference",
    "energy_improvement_vs_initial",
    "fidelity",
    "af_weight_participation",
    "evaluations",
]]

## Energy, Fidelity, and Bond Delocalization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

for source, group in df.groupby("source_file"):
    group = group.sort_values("depth")
    axes[0].plot(group["depth"], group["energy"], marker="o", label=source)
    axes[1].plot(group["depth"], group["fidelity"], marker="o", label=source)
    axes[2].plot(group["depth"], group["af_weight_participation"], marker="o", label=source)

axes[0].axhline(EXACT_ENERGY, color="black", ls="--", lw=1, label="exact")
axes[0].set_ylabel("Energy")
axes[1].set_ylabel("Fidelity")
axes[2].set_ylabel("AF bond participation")

for ax in axes:
    ax.set_xlabel("Depth p")
    ax.legend(fontsize=6)

fig.tight_layout()
plt.show()